In [7]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA, KernelPCA

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


# 1. Split and scale the data
def split_scalar(indep_X, dep_Y):

    X_train, X_test, y_train, y_test = train_test_split(
        indep_X,
        dep_Y,
        test_size=0.25,
        random_state=0,
        stratify=dep_Y
    )

    sc = StandardScaler()

    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

    return X_train, X_test, y_train, y_test


# 2. LDA
def LDA_Method(indep_X, dep_Y, n):

    X_train, X_test, y_train, y_test = split_scalar(indep_X, dep_Y)

    # LDA components cannot exceed number of classes - 1
    n_components = 1

    lda = LinearDiscriminantAnalysis(n_components=n_components)

    X_train = lda.fit_transform(X_train, y_train)
    X_test = lda.transform(X_test)

    return X_train, X_test, y_train, y_test


# 3. PCA
def PCA_Method(indep_X, dep_Y, n):

    X_train, X_test, y_train, y_test = split_scalar(indep_X, dep_Y)

    pca = PCA(n_components=n)

    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)

    return X_train, X_test, y_train, y_test


# 4. Kernel PCA
def Kernel_PCA(indep_X, dep_Y, n):

    X_train, X_test, y_train, y_test = split_scalar(indep_X, dep_Y)

    kpca = KernelPCA(
        n_components=n,
        kernel="rbf",
        eigen_solver="arpack"
    )

    X_train = kpca.fit_transform(X_train)
    X_test = kpca.transform(X_test)

    return X_train, X_test, y_train, y_test


# 5. Confusion Matrix and Accuracy
def cm_prediction(classifier, X_test, y_test):

    y_pred = classifier.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    Accuracy = accuracy_score(y_test, y_pred)

    report = classification_report(y_test, y_pred, zero_division=0)

    return classifier, Accuracy, report, X_test, y_test, cm


# 6. Common prediction function
def prediction(classifier, X_train, y_train, X_test, y_test):

    classifier.fit(X_train, y_train)

    return cm_prediction(classifier, X_test, y_test)


# 7. Logistic Regression
def logistic(X_train, y_train, X_test, y_test):

    classifier = LogisticRegression(random_state=0, max_iter=1000)

    return prediction(classifier, X_train, y_train, X_test, y_test)


# 8. SVM Linear
def svm_linear(X_train, y_train, X_test, y_test):

    classifier = SVC(kernel="linear", random_state=0)

    return prediction(classifier, X_train, y_train, X_test, y_test)


# 9. SVM Non-linear
def svm_NL(X_train, y_train, X_test, y_test):

    classifier = SVC(kernel="rbf", random_state=0)

    return prediction(classifier, X_train, y_train, X_test, y_test)


# 10. Naive Bayes
def Navie(X_train, y_train, X_test, y_test):

    classifier = GaussianNB()

    return prediction(classifier, X_train, y_train, X_test, y_test)


# 11. KNN
def knn(X_train, y_train, X_test, y_test):

    classifier = KNeighborsClassifier(
        n_neighbors=5,
        metric="minkowski",
        p=2
    )

    return prediction(classifier, X_train, y_train, X_test, y_test)


# 12. Decision Tree
def Decision(X_train, y_train, X_test, y_test):

    classifier = DecisionTreeClassifier(
        criterion="entropy",
        random_state=0
    )

    return prediction(classifier, X_train, y_train, X_test, y_test)


# 13. Random Forest
def random(X_train, y_train, X_test, y_test):

    classifier = RandomForestClassifier(
        n_estimators=10,
        criterion="entropy",
        random_state=0
    )

    return prediction(classifier, X_train, y_train, X_test, y_test)


# 14. Classification Accuracy Comparison
def lda_pca_classification(
    acclog, accsvml, accsvmnl, accknn,
    accnav, accdes, accrf
):

    dataframe = pd.DataFrame(
        index=["LDA", "PCA", "Kernel PCA"],
        columns=[
            "Logistic", "SVMl", "SVMnl",
            "KNN", "Naive", "Decision", "Random"
        ]
    )

    dataframe["Logistic"] = acclog
    dataframe["SVMl"] = accsvml
    dataframe["SVMnl"] = accsvmnl
    dataframe["KNN"] = accknn
    dataframe["Naive"] = accnav
    dataframe["Decision"] = accdes
    dataframe["Random"] = accrf

    return dataframe.astype(float)


# 15. Load Dataset
dataset1 = pd.read_csv("prep.csv", index_col=None)

df2 = dataset1.copy()

df2 = pd.get_dummies(df2, drop_first=True)

indep_X = df2.drop("classification_yes", axis=1)

dep_Y = df2["classification_yes"]


# 16. Apply LDA, PCA and Kernel PCA
X_train_lda, X_test_lda, y_train_lda, y_test_lda = LDA_Method(
    indep_X, dep_Y, 5
)

X_train_pca, X_test_pca, y_train_pca, y_test_pca = PCA_Method(
    indep_X, dep_Y, 5
)

X_train_kpca, X_test_kpca, y_train_kpca, y_test_kpca = Kernel_PCA(
    indep_X, dep_Y, 5
)


# 17. Store the three reduced datasets
reduced_data = [
    (X_train_lda, X_test_lda, y_train_lda, y_test_lda),
    (X_train_pca, X_test_pca, y_train_pca, y_test_pca),
    (X_train_kpca, X_test_kpca, y_train_kpca, y_test_kpca)
]


# 18. Create accuracy lists
acclog = []
accsvml = []
accsvmnl = []
accknn = []
accnav = []
accdes = []
accrf = []


# 19. Run all seven classifiers for each technique
for X_train, X_test, y_train, y_test in reduced_data:

    classifier, Accuracy, report, X_test, y_test, cm = logistic(
        X_train, y_train, X_test, y_test
    )
    acclog.append(Accuracy)

    classifier, Accuracy, report, X_test, y_test, cm = svm_linear(
        X_train, y_train, X_test, y_test
    )
    accsvml.append(Accuracy)

    classifier, Accuracy, report, X_test, y_test, cm = svm_NL(
        X_train, y_train, X_test, y_test
    )
    accsvmnl.append(Accuracy)

    classifier, Accuracy, report, X_test, y_test, cm = knn(
        X_train, y_train, X_test, y_test
    )
    accknn.append(Accuracy)

    classifier, Accuracy, report, X_test, y_test, cm = Navie(
        X_train, y_train, X_test, y_test
    )
    accnav.append(Accuracy)

    classifier, Accuracy, report, X_test, y_test, cm = Decision(
        X_train, y_train, X_test, y_test
    )
    accdes.append(Accuracy)

    classifier, Accuracy, report, X_test, y_test, cm = random(
        X_train, y_train, X_test, y_test
    )
    accrf.append(Accuracy)


# 20. Display comparison table
result = lda_pca_classification(
    acclog, accsvml, accsvmnl, accknn,
    accnav, accdes, accrf
)

print("\nLDA, PCA and Kernel PCA - Classification Accuracy\n")

print(result.round(3))

print("LDA components:", X_train_lda.shape[1])

print("PCA components:", X_train_pca.shape[1])

print("Kernel PCA components:", X_train_kpca.shape[1])



LDA, PCA and Kernel PCA - Classification Accuracy

            Logistic  SVMl  SVMnl   KNN  Naive  Decision  Random
LDA             0.98  0.98   0.98  0.96   0.98      0.95    0.95
PCA             0.97  0.98   0.98  0.99   0.98      0.98    0.98
Kernel PCA      0.97  0.98   0.98  0.96   0.98      0.99    0.99
LDA components: 1
PCA components: 5
Kernel PCA components: 5
